In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

processed_filepath = './datasets/processed/'
data = pd.read_csv(processed_filepath + 'Thursday.csv')

# Przygotowanie danych
X = data.drop('Label', axis=1)
y = data['Label']

# Kodowanie etykiet
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Podział na zbiory treningowe i testowe
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalizacja danych numerycznych
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Konwersja do formatu PyTorch
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

# Inicjalizacja modelu TabTransformer
model = TabNetClassifier(
    n_d=64,  # Rozmiar wewnętrznych reprezentacji
    n_a=64,  # Rozmiar uwagi
    n_steps=5,  # Liczba kroków decyzyjnych
    gamma=1.5,  # Współczynnik uwagi
    cat_idxs=[],  # Indeksy kolumn kategorycznych (jeśli istnieją)
    cat_dims=[],  # Liczba unikalnych wartości w kolumnach kategorycznych
    cat_emb_dim=1,  # Rozmiar osadzeń dla kolumn kategorycznych
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax'  # Typ maski (entmax lub sparsemax)
)

# Trening modelu
model.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_test, y_test)],
    eval_name=['test'],
    eval_metric=['accuracy', 'balanced_accuracy'],
    max_epochs=50,
    patience=10,
    batch_size=256,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

# Predykcja
y_pred_probs = model.predict_proba(X_test)[:, 1]
y_pred_classes = (y_pred_probs >= 0.5).astype(int)

# Ewaluacja
cm = confusion_matrix(y_test, y_pred_classes)
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

AttributeError: partially initialized module 'torch' has no attribute 'types' (most likely due to a circular import)